In [1]:
!free -h
!nvidia-smi --query-gpu=name,memory.total --format=csv

               total        used        free      shared  buff/cache   available
Mem:            52Gi       1.4Gi        48Gi       2.2Mi       3.8Gi        51Gi
Swap:             0B          0B          0B
name, memory.total [MiB]
NVIDIA L4, 23034 MiB


install and record versions

In [2]:
!pip install -q vllm==0.30.0 openai pillow requests

import os, io, sys, json, time, base64, subprocess, requests
import vllm, torch, transformers
from PIL import Image
from google.colab import userdata

os.environ["HF_HUB_DISABLE_XET"] = "1"
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
MODEL = "Qwen/Qwen2.5-VL-3B-Instruct"

VERSIONS = {
    "python": sys.version.split()[0],
    "vllm": vllm.__version__,
    "torch": torch.__version__,
    "cuda": torch.version.cuda,
    "transformers": transformers.__version__,
    "gpu": torch.cuda.get_device_name(0),
    "model": MODEL,
}
print(json.dumps(VERSIONS, indent=2))

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.9/87.9 kB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 314.9/314.9 MB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 109.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.7/211.7 kB 22.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.3/18.3 MB 132.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.7/322.7 kB 31.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.0/111.0 kB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.4/45.4 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 127.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 782.6/782.6 kB 57.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 108.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.2/43.2 MB 61.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7

start the vLLM server

In [3]:
SERVER_ARGS = ["--max-model-len", "16384", "--gpu-memory-utilization", "0.85", "--port", "8000"]

log = open("/content/server_gate1.log", "w")
server = subprocess.Popen(
    ["python", "-m", "vllm.entrypoints.openai.api_server", "--model", MODEL, *SERVER_ARGS],
    stdout=log, stderr=subprocess.STDOUT, env=os.environ.copy())

t0 = time.time()
while True:
    try:
        if requests.get("http://localhost:8000/health", timeout=2).status_code == 200:
            print(f"Server up in {time.time() - t0:.0f}s"); break
    except requests.exceptions.RequestException:
        pass
    if server.poll() is not None or time.time() - t0 > 900:
        print("Server failed. Last log lines:")
        print(open("/content/server_gate1.log").read()[-3000:]); break
    time.sleep(5)

Server up in 295s


prepare the stop sign image

In [4]:
from vllm.assets.image import ImageAsset

img = ImageAsset("stop_sign").pil_image.convert("RGB")
buf = io.BytesIO()
img.save(buf, format="JPEG", quality=95)
JPEG_BYTES = buf.getvalue()

DATA_URL = "data:image/jpeg;base64," + base64.b64encode(JPEG_BYTES).decode()
SENT_IMG = Image.open(io.BytesIO(JPEG_BYTES)).convert("RGB")   # exactly what vLLM will decode
W, H = SENT_IMG.size
print(f"width {W}, height {H}, JPEG {len(JPEG_BYTES)/1024:.0f} KB, base64 {len(DATA_URL)/1024:.0f} KB")

width 1300, height 876, JPEG 248 KB, base64 330 KB


gate test #1

In [5]:
from openai import OpenAI

client = OpenAI(base_url="http://localhost:8000/v1", api_key="none")
PROMPT = "Describe this image in one sentence."

def ask(messages):
    """Send one non-streaming chat request to vLLM.

    Use temperature=0 and max_tokens=60 (same as the original test #1).
    Returns (prompt_tokens, answer_text).
    """
    r = client.chat.completions.create(
        model=MODEL,
        messages=messages,
        temperature=0,
        max_tokens=60,
    )
    return r.usage.prompt_tokens, r.choices[0].message.content

img_messages = [{
    "role": "user",
    "content": [
        {"type": "image_url", "image_url": {"url": DATA_URL}},
        {"type": "text", "text": PROMPT},
    ],
}]

txt_messages = [{
    "role": "user",
    "content": PROMPT,
}]

img_prompt_tokens, answer = ask(img_messages)
txt_prompt_tokens, _ = ask(txt_messages)

measured = img_prompt_tokens - txt_prompt_tokens
expected = round(H / 28) * round(W / 28)

print("answer:", answer)
print(f"prompt tokens with image: {img_prompt_tokens}, text only: {txt_prompt_tokens}")
print(f"measured image tokens: {measured}, 28x28 rule: {expected}, gap: {measured - expected}")

answer: A stop sign is in front of a red and gold gate with Chinese characters on it.
prompt tokens with image: 1454, text only: 26
measured image tokens: 1428, 28x28 rule: 1426, gap: 2


settle the marker question

In [6]:
from transformers import AutoProcessor

proc = AutoProcessor.from_pretrained(MODEL)

def count_tokens_hf(pil_img, prompt):
    """Build the exact prompt Qwen sees for (image + prompt) and count tokens by type.

    Returns a dict: total, vision_start, image_pad, vision_end, grid.
    """
    messages = [{"role": "user", "content": [
        {"type": "image"},
        {"type": "text", "text": prompt}]}]

    text = proc.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    print(text)

    inputs = proc(text=[text], images=[pil_img], return_tensors="pt")
    ids = inputs["input_ids"][0].tolist()

    tid = proc.tokenizer.convert_tokens_to_ids
    return {
        "total": len(ids),
        "vision_start": ids.count(tid("<|vision_start|>")),
        "image_pad": ids.count(tid("<|image_pad|>")),
        "vision_end": ids.count(tid("<|vision_end|>")),
        "grid": inputs["image_grid_thw"][0].tolist(),
    }

hf = count_tokens_hf(SENT_IMG, PROMPT)
print(hf)
print("HF total == vLLM prompt_tokens:   ", hf["total"] == img_prompt_tokens)
print("image_pad == 28x28 rule:          ", hf["image_pad"] == expected)
print("start + pads + end == measured:   ", hf["vision_start"] + hf["image_pad"] + hf["vision_end"] == measured)

<|im_start|>system
You are a helpful assistant.<|im_end|>
<|im_start|>user
<|vision_start|><|image_pad|><|vision_end|>Describe this image in one sentence.<|im_end|>
<|im_start|>assistant

{'total': 1454, 'vision_start': 1, 'image_pad': 1426, 'vision_end': 1, 'grid': [1, 62, 92]}
HF total == vLLM prompt_tokens:    True
image_pad == 28x28 rule:           True
start + pads + end == measured:    True


save the result

In [7]:
from google.colab import drive
import datetime, shutil

drive.mount("/content/drive")
DEST = "/content/drive/MyDrive/visionabr_results/phase_minus1"
os.makedirs(DEST, exist_ok=True)

result = {
    "test": "gate1_model_runs_and_token_count",
    "timestamp": datetime.datetime.now(datetime.timezone.utc).isoformat(),
    "versions": VERSIONS,
    "config": {
        "server_args": SERVER_ARGS,
        "image": "vllm ImageAsset stop_sign, JPEG quality 95, base64 data URL",
        "image_size_wh": [W, H],
        "prompt": PROMPT,
        "temperature": 0,
        "max_tokens": 60,
        "max_pixels": "default",
        "streaming": False,
    },
    "raw": {
        "answer": answer,
        "prompt_tokens_with_image": img_prompt_tokens,
        "prompt_tokens_text_only": txt_prompt_tokens,
        "hf_processor_counts": hf,
    },
    "summary": {
        "measured_image_tokens": measured,
        "rule_28x28": expected,
        "gap": measured - expected,
        "markers_verified": hf["vision_start"] == 1 and hf["vision_end"] == 1
                            and hf["vision_start"] + hf["image_pad"] + hf["vision_end"] == measured,
    },
}

stamp = datetime.date.today().isoformat()
path = f"{DEST}/{stamp}_gate1_tokens_stopsign.json"
json.dump(result, open(path, "w"), indent=2)
print("saved:", path)

shutil.copy("/content/server_gate1.log", f"{DEST}/{stamp}_gate1_server.log")
print("saved:", f"{DEST}/{stamp}_gate1_server.log")

Mounted at /content/drive
saved: /content/drive/MyDrive/visionabr_results/phase_minus1/2026-10-11_gate1_tokens_stopsign.json
saved: /content/drive/MyDrive/visionabr_results/phase_minus1/2026-10-11_gate1_server.log


gate test #2

In [8]:
import math

SIZES = {"full": None, "~1024": 802816, "~512": 401408, "~256": 200704}

def predicted_image_tokens(w, h, max_pixels=None, f=28):
    """Qwen2.5-VL resize rule: keep shape, round sides to multiples of 28,
    and if over max_pixels, scale down and round DOWN. Returns (tokens incl. 2 markers, (h, w))."""
    h_bar, w_bar = round(h / f) * f, round(w / f) * f
    if max_pixels is not None and h_bar * w_bar > max_pixels:
        beta = math.sqrt(h * w / max_pixels)
        h_bar = math.floor(h / beta / f) * f
        w_bar = math.floor(w / beta / f) * f
    return (h_bar // f) * (w_bar // f) + 2, (h_bar, w_bar)

def ask_sized(max_px):
    """Same image request as gate #1, with an optional per-request max_pixels."""
    extra = {"mm_processor_kwargs": {"max_pixels": max_px}} if max_px else None
    r = client.chat.completions.create(
        model=MODEL, messages=img_messages, temperature=0, max_tokens=60, extra_body=extra)
    return r.usage.prompt_tokens, r.choices[0].message.content

gate2_rows = []
for name, px in SIZES.items():
    pt, ans = ask_sized(px)
    measured_img = pt - txt_prompt_tokens
    pred, (rh, rw) = predicted_image_tokens(W, H, px)
    gate2_rows.append({
        "size": name, "max_pixels": px, "prompt_tokens": pt,
        "image_tokens_incl_markers": measured_img, "predicted": pred,
        "resized_wh": [rw, rh], "answer": ans,
    })
    status = "OK" if measured_img == pred else "MISMATCH"
    print(f"{name:>6} | max_pixels {str(px):>7} | resized {rw}x{rh} | "
          f"image tokens {measured_img:>4} (rule {pred:>4}) {status} | {ans[:55]}")

  full | max_pixels    None | resized 1288x868 | image tokens 1428 (rule 1428) OK | A stop sign is in front of a red and gold gate with Chi
 ~1024 | max_pixels  802816 | resized 1064x728 | image tokens  990 (rule  990) OK | A stop sign is in front of a Chinese style building.
  ~512 | max_pixels  401408 | resized 756x504 | image tokens  488 (rule  488) OK | A stop sign is in front of a Chinese style archway.
  ~256 | max_pixels  200704 | resized 532x364 | image tokens  249 (rule  249) OK | A stop sign is in front of a Chinese restaurant.


save gate #2

In [9]:
gate2 = {
    "test": "gate2_per_request_max_pixels",
    "timestamp": datetime.datetime.now(datetime.timezone.utc).isoformat(),
    "versions": VERSIONS,
    "config": {
        "server_args": SERVER_ARGS,
        "image": "vllm ImageAsset stop_sign, JPEG quality 95, base64 data URL (same image every request)",
        "image_size_wh": [W, H],
        "prompt": PROMPT, "temperature": 0, "max_tokens": 60, "streaming": False,
        "text_only_prompt_tokens": txt_prompt_tokens,
        "server_restarts_between_sizes": 0,
    },
    "raw": gate2_rows,
    "summary": {
        "image_tokens": {r["size"]: r["image_tokens_incl_markers"] for r in gate2_rows},
        "all_match_resize_rule": all(r["image_tokens_incl_markers"] == r["predicted"] for r in gate2_rows),
    },
}
path2 = f"{DEST}/{stamp}_gate2_max_pixels_stopsign.json"
json.dump(gate2, open(path2, "w"), indent=2)
print("saved:", path2)

saved: /content/drive/MyDrive/visionabr_results/phase_minus1/2026-10-11_gate2_max_pixels_stopsign.json


gate test #3 pre-req work

restart the server with prefix caching off

In [10]:
server.terminate(); server.wait(); time.sleep(10)

SERVER_ARGS_G3 = SERVER_ARGS + ["--no-enable-prefix-caching"]
log3 = open("/content/server_gate3.log", "w")
server = subprocess.Popen(
    ["python", "-m", "vllm.entrypoints.openai.api_server", "--model", MODEL, *SERVER_ARGS_G3],
    stdout=log3, stderr=subprocess.STDOUT, env=os.environ.copy())

t0 = time.time()
while True:
    try:
        if requests.get("http://localhost:8000/health", timeout=2).status_code == 200:
            print(f"Server up in {time.time() - t0:.0f}s"); break
    except requests.exceptions.RequestException:
        pass
    if server.poll() is not None or time.time() - t0 > 900:
        print("Server failed. Last log lines:")
        print(open("/content/server_gate3.log").read()[-3000:]); break
    time.sleep(5)

Server up in 100s


gate #3 and image share

In [11]:
import asyncio, statistics, itertools
import numpy as np
from openai import AsyncOpenAI

aclient = AsyncOpenAI(base_url="http://localhost:8000/v1", api_key="none")
rng = np.random.default_rng(20261011)
BASE_ARR = np.array(img)
TEXT_IDS = itertools.count(100000)   # 6-digit ids, so every text prefix has the same token length

def unique_image_url():
    """Stop sign with a random 8x8 corner patch, so no two requests share an image."""
    arr = BASE_ARR.copy()
    arr[:8, :8] = rng.integers(0, 256, size=(8, 8, 3), dtype=np.uint8)
    b = io.BytesIO(); Image.fromarray(arr).save(b, format="JPEG", quality=95)
    return "data:image/jpeg;base64," + base64.b64encode(b.getvalue()).decode()

def img_factory():
    return [{"role": "user", "content": [
        {"type": "image_url", "image_url": {"url": unique_image_url()}},
        {"type": "text", "text": PROMPT}]}]

def text_factory(n_words):
    def make():
        return [{"role": "user", "content": f"[{next(TEXT_IDS)}] " + "word " * n_words + PROMPT}]
    return make

async def timed_request(messages, mm_kwargs=None):
    """Streaming request. TTFT = time until the first chunk with text."""
    extra = {"mm_processor_kwargs": mm_kwargs} if mm_kwargs else None
    t0 = time.perf_counter()
    stream = await aclient.chat.completions.create(
        model=MODEL, messages=messages, temperature=0, max_tokens=16,
        stream=True, stream_options={"include_usage": True}, extra_body=extra)
    ttft, ptoks = None, None
    async for chunk in stream:
        if ttft is None and chunk.choices and chunk.choices[0].delta.content:
            ttft = (time.perf_counter() - t0) * 1000
        if chunk.usage:
            ptoks = chunk.usage.prompt_tokens
    return {"ttft_ms": ttft, "prompt_tokens": ptoks}

async def run(factory, n, conc, mm_kwargs=None):
    """Closed loop: n requests, at most `conc` in flight. Messages are built before timing starts."""
    msgs = [factory() for _ in range(n)]
    sem = asyncio.Semaphore(conc)
    async def one(m):
        async with sem:
            return await timed_request(m, mm_kwargs)
    return await asyncio.gather(*[one(m) for m in msgs])

def stats(recs):
    t = [r["ttft_ms"] for r in recs if r["ttft_ms"] is not None]
    return {"n": len(t), "p50_ms": round(statistics.median(t), 1),
            "p95_ms": round(float(np.percentile(t, 95)), 1),
            "prompt_tokens": int(statistics.median(r["prompt_tokens"] for r in recs))}

async def match_text_length(target):
    """Find a word count whose text prompt has exactly `target` tokens."""
    n = max(1, target - 30)
    for _ in range(5):
        got = (await run(text_factory(n), 1, 1))[0]["prompt_tokens"]
        if got == target:
            break
        n += target - got
    return n, got

SETTINGS3 = {"full": None, "~1024": 802816, "~512": 401408, "~256": 200704}
N_WARMUP, N_RUNS, CONCS = 3, 20, [1, 8]

gate3 = []
for name, px in SETTINGS3.items():
    kw = {"max_pixels": px} if px else None
    row = {"size": name, "max_pixels": px, "raw": {}}

    await run(img_factory, N_WARMUP, 1, kw)                      # warm-up, discarded
    for c in CONCS:
        recs = await run(img_factory, N_RUNS, c, kw)
        row["raw"][f"img_c{c}"], row[f"img_c{c}"] = recs, stats(recs)

    target = row["img_c1"]["prompt_tokens"]
    n_words, got = await match_text_length(target)
    row["text_match"] = {"target": target, "got": got, "n_words": n_words}

    tf = text_factory(n_words)
    await run(tf, N_WARMUP, 1)                                    # warm-up, discarded
    for c in CONCS:
        recs = await run(tf, N_RUNS, c)
        row["raw"][f"txt_c{c}"], row[f"txt_c{c}"] = recs, stats(recs)

    row["image_share_c1"] = round(
        (row["img_c1"]["p50_ms"] - row["txt_c1"]["p50_ms"]) / row["img_c1"]["p50_ms"], 3)
    gate3.append(row)
    print(f"done: {name}")

print(f"\n{'size':>6} {'tokens':>6} {'txt tok':>7} | {'img c1 p50':>10} {'txt c1 p50':>10} {'share':>6} | "
      f"{'img c8 p50':>10} {'img c8 p95':>10} {'txt c8 p50':>10}")
for r in gate3:
    print(f"{r['size']:>6} {r['img_c1']['prompt_tokens']:>6} {r['text_match']['got']:>7} | "
          f"{r['img_c1']['p50_ms']:>10} {r['txt_c1']['p50_ms']:>10} {r['image_share_c1']:>6.1%} | "
          f"{r['img_c8']['p50_ms']:>10} {r['img_c8']['p95_ms']:>10} {r['txt_c8']['p50_ms']:>10}")

done: full
done: ~1024
done: ~512
done: ~256

  size tokens txt tok | img c1 p50 txt c1 p50  share | img c8 p50 img c8 p95 txt c8 p50
  full   1454    1454 |      451.1      181.5  59.8% |     2056.8     3494.0      626.3
 ~1024   1016    1016 |      330.2      135.3  59.0% |     1376.7     2598.8      534.8
  ~512    514     514 |      186.6       77.5  58.5% |     1074.6     1460.1      326.0
  ~256    275     275 |      126.4       51.6  59.2% |      653.8      897.6      296.3


save gate test #3

In [12]:
gate3_result = {
    "test": "gate3_speedup_and_image_share",
    "timestamp": datetime.datetime.now(datetime.timezone.utc).isoformat(),
    "versions": VERSIONS,
    "config": {
        "server_args": SERVER_ARGS_G3,
        "image": "stop_sign 1300x876, random 8x8 corner patch per request (numpy seed 20261011), JPEG q95",
        "text_prompt": "'[NNNNNN] ' + 'word ' * n + PROMPT, length matched to image prompt_tokens",
        "prompt": PROMPT, "temperature": 0, "max_tokens": 16, "streaming": True,
        "warmup_per_point": N_WARMUP, "measured_per_point": N_RUNS, "concurrency": CONCS,
        "loop": "closed (asyncio.Semaphore)",
        "ttft_definition": "client-side, request start to first chunk with non-empty content",
        "image_share": "(img c1 p50 - same-length text c1 p50) / img c1 p50",
    },
    "rows": gate3,
}
path3 = f"{DEST}/{datetime.date.today().isoformat()}_gate3_speedup_image_share.json"
json.dump(gate3_result, open(path3, "w"), indent=2)
print("saved:", path3)
shutil.copy("/content/server_gate3.log", path3.replace("_speedup_image_share.json", "_server.log"))
print("saved:", path3.replace("_speedup_image_share.json", "_server.log"))

saved: /content/drive/MyDrive/visionabr_results/phase_minus1/2026-10-11_gate3_speedup_image_share.json
saved: /content/drive/MyDrive/visionabr_results/phase_minus1/2026-10-11_gate3_server.log
